# agent_tools

> Learning operations that a host agent wraps as tools.

#| export
A host agent, such as Leela, makes a walk, reads its state, grades an answer and notes a confusion. This module holds those operations. It imports from `learner`, `models`, `sources`, `tutor`, `path` and `show`. Each operation takes the learner's store `lr`, and the host's model call `ask(prompt, sp='')` when it needs a model.

**The contract of every operation**

- An operation returns one dict with these fields: `status` (`'ok'`, `'error'` or `'stopped'`), `operation_id`, `vault`, `walk`, `step`, `revision`, `data` (a dict) and `error`. `result` builds it.
- `error` is `None`, or one of `ERRORS`: `'vault_locked'`, `'read_only'`, `'no_source'`, `'model_refused'`, `'withheld'`, `'stopped'`, `'not_found'`, `'conflict'`.
- An expected failure never raises. It has `status` `'error'` and an `error`, and `data['message']` says why. A stop has `status` `'stopped'` and `error` `'stopped'`.
- The values are JSON values only. The caller owns `operation_id`: the engine leaves it `None`. `vault` is the file name of the vault. `revision` is the `updated` time of the walk.
- Stop is a `stopping()` callback that returns true after a stop. The operation checks it before each model call and after each one, so before it saves anything. A stop leaves no partial walk.
- `TOOLS` lists each operation that a host makes a tool, with the arguments that the model gives. The host gives `lr`, `ask`, `stopping` and `progress` itself.

In [ ]:
#| default_exp agent_tools

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # nor the real ramabana config
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| hide
from fastcore.test import *

Import what this module uses. The model, the walk and the tutor come from the engine modules above it.

In [ ]:
#| export
import re, time
from fastcore.utils import *
from abhyasa import learner, path, show
from abhyasa.learner import Learning
from abhyasa.models import Withheld, guard
from abhyasa.sources import VaultSource, clip
from abhyasa.tutor import Tutor

## The result

`result` builds the one shape. `failed` builds an expected failure: a stop has the status `'stopped'`, and each other failure has the status `'error'`.

In [ ]:
#| export
ERRORS = ('vault_locked', 'read_only', 'no_source', 'model_refused', 'withheld', 'stopped', 'not_found', 'conflict')
STATUSES = ('ok', 'error', 'stopped')
FIELDS = ('operation_id', 'vault', 'walk', 'step', 'revision', 'data', 'error')

def result(
    status:str,   # one of `STATUSES`
    **kw          # a value for any field of `FIELDS`
) -> dict:
    "The one result shape: `status` and each field of `FIELDS`, empty unless `kw` gives it."
    if status not in STATUSES: raise ValueError(f'no status {status!r}')
    if (bad := set(kw) - set(FIELDS)): raise TypeError(f'no result field {sorted(bad)}')
    if kw.get('error') not in (None, *ERRORS): raise ValueError(f"no error {kw['error']!r}")
    return dict(status=status, operation_id=None, vault=None, walk=None, step=None, revision=None, data={}, error=None) | kw

def failed(
    error:str,       # one of `ERRORS`
    message:str='',  # why, in plain words; it goes in `data['message']`
    **kw             # a value for `walk`, `step` or another field
) -> dict:
    "An expected failure as a result."
    return result('stopped' if error == 'stopped' else 'error', error=error, data=dict(message=message), **kw)

`TOOLS` names each operation that a host makes a tool. The arguments are the ones that the model gives. They have the names of the function parameters, so a host passes them straight through.

In [ ]:
#| export
TOOLS = (
    ('create_walk', dict(goal='str: what the learner wants to learn')),
    ('walk_state', dict(jid='str: the walk id')),
    ('record_attempt', dict(jid='str: the walk id', i='int: the place of the step in the walk', answer="str: the learner's answer",
                            attempt_id='str: a key that the caller makes once for this answer; the same key with the same answer gives the same result')),
    ('note_confusion', dict(jid='str: the walk id', i='int: the place of the step in the walk', note='str: what confuses the learner, in their words')))

A failure has the status `'error'`, its `error` and a message. A stop has its own status. A name that is not in `ERRORS` is refused.

This cell checks both.

In [ ]:
test_eq((failed('not_found', 'no walk')['status'], failed('stopped')['status'], result('ok')['error']), ('error', 'stopped', None))
test_fail(lambda: failed('broken'), contains='no error')

## Errors that the contract names

`_run` turns the errors that the contract names into results. Any other error is a bug, so it stays an error.

A stop is a `BaseException`. The tutor has `except Exception` blocks, and they must not swallow it. A host model call that declines raises `ModelRefused`. A reply that is not JSON after two tries is a `ValueError` from `ask_json`, and it also counts as a refusal.

The vault has no read-only or locked flag. A failed write shows as an SQLite error, so `vault_failure` reads the class names and the message.

In [ ]:
#| export
class ModelRefused(Exception):
    "What a host `ask` raises when the model declines to answer."

class _Stopped(BaseException):
    "Ends an operation at a safe point."

def vault_failure(e:BaseException) -> str|None:
    "`'read_only'` or `'vault_locked'` when `e` is a failed write to the vault, else None."
    s = ' '.join([*(c.__name__ for c in type(e).__mro__), str(e)]).lower()
    return 'read_only' if 'readonly' in s else 'vault_locked' if re.search(r'\b(?:busy|locked)', s) else None

def _run(f, **at) -> dict:
    "The result of `f()`; `at` (`walk`, `step`) goes on a failure."
    try: return f()
    except _Stopped: return failed('stopped', 'stopped before anything was saved', **at)
    except Withheld as e: return failed('withheld', str(e), **at)
    except (ModelRefused, ValueError) as e: return failed('model_refused', str(e), **at)
    except Exception as e:
        if (k := vault_failure(e)): return failed(k, str(e), **at)
        raise

def _stoppable(ask, stopping):
    "`ask` that checks `stopping()` before the call and after it, so before its answer is used."
    def f(prompt, sp=''):
        if stopping(): raise _Stopped()
        out = ask(prompt, sp)
        if stopping(): raise _Stopped()
        return out
    return f

def _vault(lr:Learning) -> str: return Path(str(lr.v.path)).name

A real SQLite error and a real apsw error give the two vault failures. Other errors give none. The word `blocked` is not `locked`.

In [ ]:
import apsw, sqlite3
test_eq([vault_failure(e) for e in (apsw.ReadOnlyError('x'), apsw.BusyError('x'), sqlite3.OperationalError('attempt to write a readonly database'),
                                     sqlite3.OperationalError('database is locked'), KeyError('blocked'))],
        ['read_only', 'vault_locked', 'read_only', 'vault_locked', None])

## Make a walk

`create_walk` calls `walk_from_topic` with the host's `ask`. It reports three stages to `progress`, with the part done from 0 to 1:

- `'planning'` goes out before the model call.
- `'sources'` goes out at each search for the passage of a step.
- `'steps'` goes out when the steps are saved.

`stopping()` is checked before and after each model call, and before each search. The planner saves the walk before it looks for passages, so a stop in that stage drops the new walk. A stop leaves no partial walk.

The goal first passes `guard`. A goal with personal data goes to the model redacted, and `data['redacted']` is true. The walk keeps the goal that the learner wrote as its title. A goal that the guard refuses gives `withheld`, and no model call is made.

In [ ]:
#| export
class _Reporting(VaultSource):
    "A `VaultSource` that tells `progress` about each search, and stops at a search after a stop."
    def __init__(self, lr:Learning, progress, stopping):
        super().__init__(lr)
        self.progress, self.stopping, self.seen = progress, stopping, 0
    def search(self, q:str, k:int=4):
        if self.stopping(): raise _Stopped()
        self.seen += 1
        self.progress('sources', min(.4 + .5 * self.seen / max(learner.val('walks.steps'), 1), .9))
        return super().search(q, k)

def create_walk(
    lr:Learning,                         # the learner's store
    ask,                                 # the host model call, `ask(prompt, sp='')`
    goal:str,                            # what the learner wants to learn
    stopping=lambda: False,              # true after a stop
    progress=lambda stage, frac: None,   # told the stage ('planning', 'sources', 'steps') and the part done
    settings:dict|None=None,             # the host's values of the settings rows (row name to value), for this call only; None reads the settings file
) -> dict:
    "Plan a walk to `goal` with the host model, find a passage for each step, and save it."
    def make():
        sent = guard(goal)
        before = {j.id for j in lr.journeys()}
        progress('planning', 0.)
        try: w = path.walk_from_topic(lr, _stoppable(ask, stopping), sent, sources=[_Reporting(lr, progress, stopping)])
        except _Stopped:
            for j in lr.journeys():
                if j.id not in before: lr.drop_journey(j.id)
            raise
        if not w.steps:
            lr.drop_journey(w.id)
            return failed('no_source', 'the plan had no step to build a walk on')
        progress('steps', 1.)
        if (redacted := sent != goal): w = lr.merge_journey(w.id, title=goal)
        return result('ok', walk=w.id, vault=_vault(lr), revision=w.get('updated'), data=dict(title=w.title, steps=len(w.steps), redacted=redacted))
    with learner.use_settings(settings): return _run(make)

## Read a walk

`walk_state` gives the steps of a walk as plain data. A step has a `quote` only when its stored passage has text and a link. The text is the stored source text, not a model summary. A step without one has `quote: None` and `ungrounded: True`.

The `gauge` is the engine's own `understanding`: depth, state and recall. This module adds no score. A step that is done, skipped or known has no `check`.

In [ ]:
#| export
def _gauge(c) -> dict:
    "What the engine knows of concept `c`: its state, its depth and its recall."
    u = learner.understanding(c)
    return dict(state=u.state, floor=u.floor, recall=round(u.recall, 2))

def _quote(s) -> dict|None:
    "The stored passage of step `s` with its link, or None when it has no text or no link."
    r = s.get('read') or {}
    return dict(text=r['text'], url=r['url'], title=r.get('title') or '') if (r.get('text') or '').strip() and r.get('url') else None

def _asked(lr:Learning, s) -> str:
    "The question of the check of step `s`: the open question, else the question of the plan, else ''."
    m = lr.open_q(s.get('qid') or '')
    return m.q.get('question', '') if m and m.status == 'open' else ' '.join(str((s.get('check') or {}).get('q') or '').split())

def _step(lr:Learning, i:int, s, c) -> dict:
    "Step `i` of a walk, as `walk_state` gives it; `s` has its `dot`, and `c` is its concept."
    q = _quote(s)
    return dict(i=i, slug=s['slug'], name=(c or {}).get('name') or s['slug'], statement=(c or {}).get('statement', ''), status=s.dot,
                quote=q, ungrounded=q is None,
                check=None if s.dot in ('done', 'skipped', 'known') else dict(question=_asked(lr, s), kind=(s.get('check') or {}).get('kind', 'short')))

def walk_state(
    lr:Learning,  # the learner's store
    jid:str,      # the walk id
    settings:dict|None=None   # the host's values of the settings rows, for this call only; None reads the settings file
) -> dict:
    "The walk as plain data: its steps with their quote or `ungrounded` and their checks, and the gauge of each concept."
    def get():
        j = lr.journey(jid)
        if j is None or j.get('mode') != 'walk': return failed('not_found', f'no walk {jid!r}', walk=jid)
        by = {c.slug: c for c in lr.concepts()}
        ws = learner.walk_state(j, by)
        steps = [_step(lr, i, s, by.get(s['slug'])) for i, s in enumerate(ws.steps)]
        return result('ok', walk=jid, vault=_vault(lr), revision=j.get('updated'),
                      data=dict(id=jid, title=j.title, steps=steps, current=ws.current, gauge={s['slug']: _gauge(by[s['slug']]) for s in steps if s['slug'] in by}))
    with learner.use_settings(settings): return _run(get, walk=jid)

## Answer a check

`record_attempt` uses the grading path of the web walk: `path.open_short` opens the check with `Tutor.ask_on`, and `Tutor.answer` grades it and saves the attempt with `Learning.record`. This module has no grader of its own.

The `attempt_id` goes in the meta of the question, so `Learning.record` keeps it on the attempt. A key is read on (walk, step). The same key with the same answer gives the stored result, with no new model call. The same key with another answer is a `conflict`. A step that is already answered is also a `conflict`.

The result has the gauge of the concept after the answer.

In [ ]:
#| export
def _attempt(lr:Learning, slug:str, jid:str, i:int, attempt_id:str):
    "The saved attempt with `attempt_id` on step `i` of walk `jid`, or None."
    return first(a for a in lr.attempts(slug, limit=None) if a.get('attempt_id') == attempt_id and a.get('walk') == jid and a.get('step') == i)

def _settle(lr:Learning, jid:str, slug:str, a) -> None:
    "Mark the step done with the outcome of attempt `a`, when `a` closed it and the step does not have it yet."
    s = first(s for s in lr.journey(jid).steps if s['slug'] == slug)
    if a.get('final') and not s.get('done'): lr.walk_slug(jid, slug, done=dict(outcome=a.outcome, aid=a.id, at=time.time()))

def _answered(lr:Learning, jid:str, i:int, slug:str, a) -> dict:
    "The result of the saved attempt `a`, the same on the first call and on a repeat."
    return result('ok', walk=jid, step=i, vault=_vault(lr), revision=lr.journey(jid).get('updated'),
                  data=dict(attempt_id=a.attempt_id, done=bool(a.get('final')), correct=bool(a.correct), outcome=a.get('outcome') or '', feedback=a.feedback,
                            first_error=a.first_error, misconception=a.misconception, concept=slug, gauge={slug: _gauge(lr.concept(slug))}))

def record_attempt(
    lr:Learning,     # the learner's store
    jid:str,         # the walk id
    i:int,           # the place of the step in the walk
    answer:str,      # the learner's answer
    attempt_id:str,  # the key of this answer, made once by the caller
    ask,             # the host model call, `ask(prompt, sp='')`
    settings:dict|None=None,   # the host's values of the settings rows, for this call only; None reads the settings file
) -> dict:
    "Grade the answer to the short check of step `i` and save the attempt, once for each `attempt_id`."
    def go():
        j = lr.journey(jid)
        if j is None or j.get('mode') != 'walk' or not 0 <= i < len(j.steps): return failed('not_found', f'no step {i} in walk {jid!r}', walk=jid, step=i)
        s = j.steps[i]; slug = s['slug']
        if (a := _attempt(lr, slug, jid, i, attempt_id)):
            if a.answer != answer: return failed('conflict', 'this attempt_id has another answer', walk=jid, step=i)
            _settle(lr, jid, slug, a)
            return _answered(lr, jid, i, slug, a)
        if s['check'].get('kind', 'short') != 'short': return failed('not_found', 'this step has no short check to answer', walk=jid, step=i)
        if s.get('done'): return failed('conflict', 'this step is already answered', walk=jid, step=i)
        tutor = Tutor(lr, ask)
        if (qid := s.get('qid')) not in tutor.open: qid = path.open_short(tutor, j, i, f'agent:{jid}').id
        tutor.open[qid]['meta']['attempt_id'] = attempt_id
        tutor.answer(qid, answer)
        a = _attempt(lr, slug, jid, i, attempt_id)
        _settle(lr, jid, slug, a)
        return _answered(lr, jid, i, slug, a)
    with learner.keyed_lock('agent-answer', (jid, i)), learner.use_settings(settings): return _run(go, walk=jid, step=i)

## Note a confusion

`note_confusion` keeps what confuses the learner in the profile, in the line `Name: note` that `show` writes for "Not clear" feedback. The tutor reads that line when it writes the next question and lesson. A confusion is not an answer: it makes no attempt and does not change the concept. The text goes through `clean_note` first. A note with no text records nothing.

In [ ]:
#| export
def note_confusion(
    lr:Learning,  # the learner's store
    jid:str,      # the walk id
    i:int,        # the place of the step in the walk
    note:str      # what confuses the learner, in their words
) -> dict:
    "Keep what confuses the learner on step `i`; it is no answer and changes no concept."
    def go():
        j = lr.journey(jid)
        if j is None or j.get('mode') != 'walk' or not 0 <= i < len(j.steps): return failed('not_found', f'no step {i} in walk {jid!r}', walk=jid, step=i)
        slug = j.steps[i]['slug']
        line = show.clean_note(note)
        kept = bool(line) and bool(show._confuses(lr, (lr.concept(slug) or {}).get('name') or slug, line))
        return result('ok', walk=jid, step=i, vault=_vault(lr), revision=j.get('updated'), data=dict(concept=slug, recorded=kept, confuses=list(lr.profile().confuses)))
    return _run(go, walk=jid, step=i)

## Context for the agent

`learner_context` is a short block of plain text for a system prompt: the open walk and step, its gauge, and the rule for sources. The rule comes first, so a cut at `n` characters never removes it. With no `i`, the step is the current one.

In [ ]:
#| export
RULE = 'Sources: quote only the text that a step shows as its quote, with its link. A step marked ungrounded has no source. Say so, and never invent a quote.'

def learner_context(
    lr:Learning,   # the learner's store
    jid:str=None,  # the open walk, if any
    i:int=None,    # the open step; the current step when None
    n:int=1200     # the most characters
) -> str:
    "A short plain-text block for a system prompt: the open walk and step, its gauge, and the rule for sources."
    r = walk_state(lr, jid) if jid else None
    if not r or r['status'] != 'ok': return clip(f'Learn mode. No walk is open.\n{RULE}', n)
    d = r['data']; k = d['current'] if i is None else i
    head = f"Learn mode. Walk: {d['title']} ({jid})."
    if k is None or not 0 <= k < len(d['steps']): body = [head, 'No step is open.']
    else:
        s = d['steps'][k]; g = d['gauge'].get(s['slug'], {})
        body = [f"{head} Step {k + 1} of {len(d['steps'])}: {s['name']}. {s['statement']}",
                f"Status: {s['status']}. Understanding: {g.get('state', 'unseen')}, depth {g.get('floor', 0)}, recall {g.get('recall', 0):.0%}.",
                f"Source: \"{clip(q['text'], 300)}\" ({q['title']}, {q['url']})" if (q := s['quote']) else 'Source: none. This step is ungrounded.',
                *([f"Check: {s['check']['question']}"] if s['check'] and s['check']['question'] else [])]
    return clip('\n'.join([RULE, *body]), n)